In [1]:
import torch

from torch_openreml import REML
from torch_openreml.covariance import (
    DummyMatrix,
    ScalarMatrix,
    CovariancePropagation,
    Sum,
)

n, p = 50, 2

y = torch.randn(n)
X = torch.randn(n, p)

Z = DummyMatrix(["a", "b"] * 25)()

V = Sum(
    CovariancePropagation(
        Z,
        ScalarMatrix(2),
    ),
    ScalarMatrix(n),
)

reml = REML(v_builder=V)

theta_start = torch.zeros(V.num_free_params)

theta_hat, beta_hat, n_iter = reml.optimize(
    y,
    X,
    theta_start,
    verbose=2,
)

 ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ 201.36it/s

Iter 1:  ⏱ 00:00 | ⚡ 181.45it/s

Iter 2:  ⏱ 00:00 | ⚡ 131.54it/s

...

Iter 4:  ⏱ 00:00 | ⚡ 327.46it/s

Iter 4:  ⏱ 00:00 | ⚡ 314.32it/s

Iter 4:  ⏱ 00:00 | ⚡ 304.41it/s

Iter 4:  ⏱ 00:00 | ⚡ 300.53it/s


∥∇∥:      11.6700, ∥Δ∥: 20.1473, η: 1.00, ∥Δᶜ∥: 20.1473, log 𝓛: -24.4980
∥∇∥:       1.9137, ∥Δ∥: 0.0192, η: 1.00, ∥Δᶜ∥: 0.0192, log 𝓛: -20.9836 (+3.5145)
∥∇∥:       0.0362, ∥Δ∥: 0.0004, η: 1.00, ∥Δᶜ∥: 0.0004, log 𝓛: -20.9650 (+0.0186)
∥∇∥:       0.0000, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, log 𝓛: -20.9650 (+0.0000)

[∇: score, Δ: 𝐉⁻¹∇, η: learning rate, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 4


In [2]:
theta_last = reml.get_theta(select="last")
theta_best = reml.get_theta(select="best")

beta_last = reml.get_beta(select="last")
beta_best = reml.get_beta(select="best")

In [3]:
beta_hat = reml.blue(y, X, theta_hat)

In [4]:
b_hat = reml.blup(
    y,
    X,
    Z,
    theta_hat,
    map_theta_to_g=ScalarMatrix(2),
    mask_theta_to_g=torch.tensor([True, False]),
)

In [5]:
y_hat_marginal = reml.marginal_predict(
    y,
    X,
    theta_hat,
)

y_hat_conditional = reml.predict(
    y,
    X,
    Z,
    theta_hat,
    map_theta_to_g=ScalarMatrix(2),
    mask_theta_to_g=torch.tensor([True, False]),
)

In [6]:
e_marginal = reml.marginal_residual(
    y,
    X,
    theta_hat,
)

e_conditional = reml.residual(
    y,
    X,
    Z,
    theta_hat,
    map_theta_to_g=ScalarMatrix(2),
    mask_theta_to_g=torch.tensor([True, False]),
)

In [7]:
loglik = reml.loglik(y, X, theta_hat)

In [8]:
import torch

from torch_openreml import REML
from torch_openreml.utils import augment, n_distinct

from torch_openreml.covariance import (
    DummyMatrix,
    IdentityMatrix,
    ScalarMatrix,
    Sum,
    CovariancePropagation,
    KroneckerProduct,
)

from torch_openreml.example_data import john_alpha

# --- response ---
y = torch.tensor(john_alpha["yield"].values)

# --- fixed effects ---
X = augment(
    torch.ones(len(john_alpha), 1),
    DummyMatrix(john_alpha["rep"], drop_first=True)()
)

# --- random effect design matrices ---
Z_gen = DummyMatrix(john_alpha["gen"])
Z_rep_block = DummyMatrix(john_alpha["rep"], john_alpha["block"])

# --- covariance components ---
G_gen = ScalarMatrix(n_distinct(john_alpha["gen"]))
G_rep = IdentityMatrix(n_distinct(john_alpha["rep"]))
G_block = ScalarMatrix(n_distinct(john_alpha["block"]))

R = ScalarMatrix(len(john_alpha))

# --- marginal covariance ---
V = Sum(
    CovariancePropagation(Z_gen, G_gen),
    CovariancePropagation(
        Z_rep_block,
        KroneckerProduct(G_rep, G_block)
    ),
    R
)

# --- REML fit ---
reml = REML(v_builder=V)

theta_start = torch.zeros(V.num_free_params)

theta_hat, beta_hat, n_iter = reml.optimize(
    y,
    X,
    theta_start,
    verbose=2,
)

# --- results ---
print("theta:", theta_hat)

print("variance components:", V.build_params(theta_hat))

print("fixed effects:", beta_hat)

print("loglik:", reml.loglik(y, X, theta_hat))

 ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ ?it/s

Iter 1:  ⏱ 00:00 | ⚡ 467.28it/s

Iter 1:  ⏱ 00:00 | ⚡ 372.43it/s

Iter 2:  ⏱ 00:00 | ⚡ 188.82it/s

...

Iter 16:  ⏱ 00:00 | ⚡ 415.93it/s

Iter 16:  ⏱ 00:00 | ⚡ 410.15it/s

Iter 16:  ⏱ 00:00 | ⚡ 404.56it/s

Iter 16:  ⏱ 00:00 | ⚡ 402.82it/s


∥∇∥:      41.8197, ∥Δ∥: 8.9438, η: 1.00, ∥Δᶜ∥: 8.9438, log 𝓛: -34.3129
∥∇∥:  315227.4375, ∥Δ∥: 0.8838, η: 1.00, ∥Δᶜ∥: 0.8838, log 𝓛: -201185.3906 (-201151.0777)
∥∇∥:  119229.9453, ∥Δ∥: 0.8520, η: 1.00, ∥Δᶜ∥: 0.8520, log 𝓛: -71844.4531 (+129340.9375)
∥∇∥:   44273.4570, ∥Δ∥: 0.8143, η: 1.00, ∥Δᶜ∥: 0.8143, log 𝓛: -26065.4551 (+45778.9980)
∥∇∥:   16330.5117, ∥Δ∥: 0.7587, η: 1.00, ∥Δᶜ∥: 0.7587, log 𝓛: -9457.1396 (+16608.3154)
∥∇∥:    6001.4102, ∥Δ∥: 0.7142, η: 1.00, ∥Δᶜ∥: 0.7142, log 𝓛: -3392.8696 (+6064.2700)
∥∇∥:    2194.8989, ∥Δ∥: 0.7031, η: 1.00, ∥Δᶜ∥: 0.7031, log 𝓛: -1181.2805 (+2211.5891)
∥∇∥:     793.7851, ∥Δ∥: 0.6952, η: 1.00, ∥Δᶜ∥: 0.6952, log 𝓛: -382.4908 (+798.7897)
∥∇∥:     278.6875, ∥Δ∥: 0.6715, η: 1.00, ∥Δᶜ∥: 0.6715, log 𝓛: -102.2992 (+280.1916)
∥∇∥:      90.4116, ∥Δ∥: 0.5957, η: 1.00, ∥Δᶜ∥: 0.5957, log 𝓛: -11.4274 (+90.8718)
∥∇∥:      23.9216, ∥Δ∥: 0.4006, η: 1.00, ∥Δᶜ∥: 0.4006, log 𝓛:  12.6792 (+24.1066)
∥∇∥:       3.8674, ∥Δ∥: 0.1392, η: 1.00, ∥Δᶜ∥: 0.1392, log 𝓛:  16.5908

In [9]:
scores = [
    torch.norm(s).item()
    for s in reml.history["score"]
]

logliks = [
    ll.item()
    for ll in reml.history["loglik"]
]

print(
    "Score norms:",
    [f"{s:.6f}" for s in scores],
)

print(
    "Log-likelihoods:",
    [f"{ll:.4f}" for ll in logliks],
)

Score norms: ['41.819736', '315227.437500', '119229.945312', '44273.457031', '16330.511719', '6001.410156', '2194.898926', '793.785095', '278.687469', '90.411598', '23.921585', '3.867368', '0.244757', '0.010371', '0.000418', '0.000021']
Log-likelihoods: ['-34.3129', '-201185.3906', '-71844.4531', '-26065.4551', '-9457.1396', '-3392.8696', '-1181.2805', '-382.4908', '-102.2992', '-11.4274', '12.6792', '16.5908', '16.8078', '16.8099', '16.8100', '16.8099']
